# TextImputer — Minimal Demo

We take a fixed input sentence, inspect its tokens, apply masking strategies manually,
and then run the TextImputer to see how coalitions map to model scores.|

In [1]:
from __future__ import annotations

import numpy as np

from shapiq.imputer.text_imputer import TextImputer

## 1. Original input & tokens

In [2]:
# to replace with our demo LLM
MODEL = "distilbert-base-uncased-finetuned-sst-2-english"

TEXT = "The acting was absolutely brilliant but the plot was painfully dull."

imputer = TextImputer(MODEL, TEXT, mask_strategy="mask")

# show individual tokens
token_strings = imputer._tokenizer.convert_ids_to_tokens(imputer._tokens)
print(f"Input text : {TEXT}")
print(f"Num tokens : {imputer.n_features}")
print()
print(f"{'idx':>4}  {'token_id':>10}  token")
print("-" * 30)
for i, (tid, tok) in enumerate(zip(imputer._tokens, token_strings, strict=False)):
    print(f"{i:>4}  {tid:>10}  {tok}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Input text : The acting was absolutely brilliant but the plot was painfully dull.
Num tokens : 12

 idx    token_id  token
------------------------------
   0        1996  the
   1        3772  acting
   2        2001  was
   3        7078  absolutely
   4        8235  brilliant
   5        2021  but
   6        1996  the
   7        5436  plot
   8        2001  was
   9       16267  painfully
  10       10634  dull
  11        1012  .


## 2. Manual masking — inspect a coalition

Here we manually construct a coalition (True = token is **present**) and see
what text gets fed into the model.|

In [3]:
n = imputer.n_features

# keep only the first half of the sentence
coalition_first_half = np.array([True] * (n // 2) + [False] * (n - n // 2))

# keep only the second half
coalition_second_half = np.array([False] * (n // 2) + [True] * (n - n // 2))


def show_coalition(imputer, coalition, label):
    tokens = imputer._tokens.copy()
    tokens[~coalition] = imputer._mask_token_id
    masked_text = imputer._decode(tokens)
    print(f"{label}")
    print(f"  coalition : {coalition.astype(int).tolist()}")
    print(f"  masked    : {masked_text}")
    print()


show_coalition(imputer, coalition_first_half, "First-half coalition")
show_coalition(imputer, coalition_second_half, "Second-half coalition")

First-half coalition
  coalition : [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0]
  masked    : the acting was absolutely brilliant but [MASK] [MASK] [MASK] [MASK] [MASK] [MASK]

Second-half coalition
  coalition : [0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1]
  masked    : [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] the plot was painfully dull.



## 3. Value function — scores for each coalition

Score > 0 → POSITIVE, score < 0 → NEGATIVE (sign-flipped from raw).

In [4]:
coalitions = np.stack(
    [
        np.ones(n, dtype=bool),  # all tokens present  → full prediction
        np.zeros(n, dtype=bool),  # no tokens           → empty prediction
        coalition_first_half,
        coalition_second_half,
    ]
)

scores = imputer.value_function(coalitions)
labels = ["Full", "Empty (all [MASK])", "First half", "Second half"]

print(f"{'Coalition':<25}  {'Score':>8}")
print("-" * 38)
for label, score in zip(labels, scores, strict=False):
    sentiment = "POSITIVE" if score > 0 else "NEGATIVE"
    print(f"{label:<25}  {score:>+8.4f}  ({sentiment})")

print()
print(f"empty_prediction = {imputer.empty_prediction:+.4f}")

Coalition                     Score
--------------------------------------
Full                        -0.9997  (NEGATIVE)
Empty (all [MASK])          -0.6274  (NEGATIVE)
First half                  +0.9822  (POSITIVE)
Second half                 -0.9997  (NEGATIVE)

empty_prediction = -0.6274


## 4. Run KernelSHAP via shapiq

Pass the imputer directly to shapiq's explainer.
The imputer acts as the value oracle — shapiq handles coalition sampling.

In [8]:
class TextImputerGame:
    def __init__(self, imputer):
        self.imputer = imputer
        self.n = imputer.n_features

    def __call__(self, coalitions):
        return self.imputer.value_function(coalitions)


game = TextImputerGame(imputer)

In [6]:
from shapiq.game_theory.exact import ExactComputer

computer = ExactComputer(game)

sv = computer(index="SV")

ValueError: n_players must be specified if game is not a Game object.

In [ ]:
print("Shapley values per token:")
print(f"{'token':<20}  {'SV':>8}")
print("-" * 32)

for tok, val in zip(token_strings, sv.values, strict=False):
    bar = "█" * int(abs(val) * 100)
    sign = "+" if val >= 0 else "-"
    print(f"{tok:<20}  {val:>+8.4f}  {sign}{bar}")